In [16]:
import json
import os
target_subject='Action'

os.chdir("/home/aiscuser/REC_Inference/LLaMA2-Accessory")
pred_output_path=f'output_subjects/{target_subject}/pred_bboxes.json'
with open(pred_output_path) as f:
    pred_results = json.load(f)

In [17]:
from copy import deepcopy

def postprocess_box(box, ori_w, ori_h):
    if ori_w == ori_h:
        return box
    if ori_w > ori_h:
        x1, y1, x2, y2 = box
        y1 -= (ori_w - ori_h) // 2
        y2 -= (ori_w - ori_h) // 2
        box = x1, y1, x2, y2
        return box
    x1, y1, x2, y2 = box
    x1 -= (ori_h - ori_w) // 2
    x2 -= (ori_h - ori_w) // 2
    box = x1, y1, x2, y2
    return box

json_path='/home/aiscuser/datasets/rec_human/person_annts_val_ready_label_v2.json'
with open(json_path) as f:
    all_annts = json.load(f)
subjects_annts=[]

for annt in all_annts:
    categories=annt['labels_v2_categories']
    if target_subject in categories:
        subjects_annts.append(annt)
all_annts=subjects_annts

print(len(all_annts),len(pred_results))

gt_bboxes=[]
pred_bboxes=[]

for annt,pred in zip(all_annts,pred_results):
    # if(annt['annt_id']!=pred['annt_id']):
    #     print('image_id mismatch')
    #     break
    assert annt['annt_id']==pred['annt_id']
    gt_bbox=deepcopy(annt['bbox'])
    # xywh to xyxy
    gt_bbox[2]+=gt_bbox[0]
    gt_bbox[3]+=gt_bbox[1]
    gt_bboxes.append(gt_bbox)

    height, width = annt['height'], annt['width']

    pred_bbox=deepcopy(pred['pred_bbox'])
    # print(pred_bbox)
    # max_ori=max(height, width)
    # x1, y1, x2, y2 = pred_bbox
    # square_bbox = round(x1 /width* max_ori,3), round(y1/height * max_ori,3), round(x2/width * max_ori,3), round(y2/height * max_ori)
    # pred_bbox = postprocess_box(square_bbox, width, height)
    # print(pred_bbox)
    pred_bboxes.append(pred_bbox)
    # break


6842 6842


In [18]:
import torch

def fp16_clamp(x, min=None, max=None):
    if not x.is_cuda and x.dtype == torch.float16:
        # clamp for cpu float16, tensor fp16 has no clamp implementation
        return x.float().clamp(min, max).half()

    return x.clamp(min, max)

def bbox_overlaps(bboxes1, bboxes2, mode='iou', is_aligned=False, eps=1e-6):
    """Calculate overlap between two set of bboxes.

    FP16 Contributed by https://github.com/open-mmlab/mmdetection/pull/4889
    Note:
        Assume bboxes1 is M x 4, bboxes2 is N x 4, when mode is 'iou',
        there are some new generated variable when calculating IOU
        using bbox_overlaps function:

        1) is_aligned is False
            area1: M x 1
            area2: N x 1
            lt: M x N x 2
            rb: M x N x 2
            wh: M x N x 2
            overlap: M x N x 1
            union: M x N x 1
            ious: M x N x 1

            Total memory:
                S = (9 x N x M + N + M) * 4 Byte,

            When using FP16, we can reduce:
                R = (9 x N x M + N + M) * 4 / 2 Byte
                R large than (N + M) * 4 * 2 is always true when N and M >= 1.
                Obviously, N + M <= N * M < 3 * N * M, when N >=2 and M >=2,
                           N + 1 < 3 * N, when N or M is 1.

            Given M = 40 (ground truth), N = 400000 (three anchor boxes
            in per grid, FPN, R-CNNs),
                R = 275 MB (one times)

            A special case (dense detection), M = 512 (ground truth),
                R = 3516 MB = 3.43 GB

            When the batch size is B, reduce:
                B x R

            Therefore, CUDA memory runs out frequently.

            Experiments on GeForce RTX 2080Ti (11019 MiB):

            |   dtype   |   M   |   N   |   Use    |   Real   |   Ideal   |
            |:----:|:----:|:----:|:----:|:----:|:----:|
            |   FP32   |   512 | 400000 | 8020 MiB |   --   |   --   |
            |   FP16   |   512 | 400000 |   4504 MiB | 3516 MiB | 3516 MiB |
            |   FP32   |   40 | 400000 |   1540 MiB |   --   |   --   |
            |   FP16   |   40 | 400000 |   1264 MiB |   276MiB   | 275 MiB |

        2) is_aligned is True
            area1: N x 1
            area2: N x 1
            lt: N x 2
            rb: N x 2
            wh: N x 2
            overlap: N x 1
            union: N x 1
            ious: N x 1

            Total memory:
                S = 11 x N * 4 Byte

            When using FP16, we can reduce:
                R = 11 x N * 4 / 2 Byte

        So do the 'giou' (large than 'iou').

        Time-wise, FP16 is generally faster than FP32.

        When gpu_assign_thr is not -1, it takes more time on cpu
        but not reduce memory.
        There, we can reduce half the memory and keep the speed.

    If ``is_aligned`` is ``False``, then calculate the overlaps between each
    bbox of bboxes1 and bboxes2, otherwise the overlaps between each aligned
    pair of bboxes1 and bboxes2.

    Args:
        bboxes1 (Tensor): shape (B, m, 4) in <x1, y1, x2, y2> format or empty.
        bboxes2 (Tensor): shape (B, n, 4) in <x1, y1, x2, y2> format or empty.
            B indicates the batch dim, in shape (B1, B2, ..., Bn).
            If ``is_aligned`` is ``True``, then m and n must be equal.
        mode (str): "iou" (intersection over union), "iof" (intersection over
            foreground) or "giou" (generalized intersection over union).
            Default "iou".
        is_aligned (bool, optional): If True, then m and n must be equal.
            Default False.
        eps (float, optional): A value added to the denominator for numerical
            stability. Default 1e-6.

    Returns:
        Tensor: shape (m, n) if ``is_aligned`` is False else shape (m,)

    Example:
        >>> bboxes1 = torch.FloatTensor([
        >>>     [0, 0, 10, 10],
        >>>     [10, 10, 20, 20],
        >>>     [32, 32, 38, 42],
        >>> ])
        >>> bboxes2 = torch.FloatTensor([
        >>>     [0, 0, 10, 20],
        >>>     [0, 10, 10, 19],
        >>>     [10, 10, 20, 20],
        >>> ])
        >>> overlaps = bbox_overlaps(bboxes1, bboxes2)
        >>> assert overlaps.shape == (3, 3)
        >>> overlaps = bbox_overlaps(bboxes1, bboxes2, is_aligned=True)
        >>> assert overlaps.shape == (3, )

    Example:
        >>> empty = torch.empty(0, 4)
        >>> nonempty = torch.FloatTensor([[0, 0, 10, 9]])
        >>> assert tuple(bbox_overlaps(empty, nonempty).shape) == (0, 1)
        >>> assert tuple(bbox_overlaps(nonempty, empty).shape) == (1, 0)
        >>> assert tuple(bbox_overlaps(empty, empty).shape) == (0, 0)
    """

    assert mode in ['iou', 'iof', 'giou'], f'Unsupported mode {mode}'
    # Either the boxes are empty or the length of boxes' last dimension is 4
    assert (bboxes1.size(-1) == 4 or bboxes1.size(0) == 0)
    assert (bboxes2.size(-1) == 4 or bboxes2.size(0) == 0)

    # Batch dim must be the same
    # Batch dim: (B1, B2, ... Bn)
    assert bboxes1.shape[:-2] == bboxes2.shape[:-2]
    batch_shape = bboxes1.shape[:-2]

    rows = bboxes1.size(-2)
    cols = bboxes2.size(-2)
    if is_aligned:
        assert rows == cols

    if rows * cols == 0:
        if is_aligned:
            return bboxes1.new(batch_shape + (rows, ))
        else:
            return bboxes1.new(batch_shape + (rows, cols))

    area1 = (bboxes1[..., 2] - bboxes1[..., 0]) * (
        bboxes1[..., 3] - bboxes1[..., 1])
    area2 = (bboxes2[..., 2] - bboxes2[..., 0]) * (
        bboxes2[..., 3] - bboxes2[..., 1])

    if is_aligned:
        lt = torch.max(bboxes1[..., :2], bboxes2[..., :2])  # [B, rows, 2]
        rb = torch.min(bboxes1[..., 2:], bboxes2[..., 2:])  # [B, rows, 2]

        wh = fp16_clamp(rb - lt, min=0)
        overlap = wh[..., 0] * wh[..., 1]

        if mode in ['iou', 'giou']:
            union = area1 + area2 - overlap
        else:
            union = area1
        if mode == 'giou':
            enclosed_lt = torch.min(bboxes1[..., :2], bboxes2[..., :2])
            enclosed_rb = torch.max(bboxes1[..., 2:], bboxes2[..., 2:])
    else:
        lt = torch.max(bboxes1[..., :, None, :2],
                       bboxes2[..., None, :, :2])  # [B, rows, cols, 2]
        rb = torch.min(bboxes1[..., :, None, 2:],
                       bboxes2[..., None, :, 2:])  # [B, rows, cols, 2]

        wh = fp16_clamp(rb - lt, min=0)
        overlap = wh[..., 0] * wh[..., 1]

        if mode in ['iou', 'giou']:
            union = area1[..., None] + area2[..., None, :] - overlap
        else:
            union = area1[..., None]
        if mode == 'giou':
            enclosed_lt = torch.min(bboxes1[..., :, None, :2],
                                    bboxes2[..., None, :, :2])
            enclosed_rb = torch.max(bboxes1[..., :, None, 2:],
                                    bboxes2[..., None, :, 2:])

    eps = union.new_tensor([eps])
    union = torch.max(union, eps)
    ious = overlap / union
    if mode in ['iou', 'iof']:
        return ious
    # calculate gious
    enclose_wh = fp16_clamp(enclosed_rb - enclosed_lt, min=0)
    enclose_area = enclose_wh[..., 0] * enclose_wh[..., 1]
    enclose_area = torch.max(enclose_area, eps)
    gious = ious - (enclose_area - union) / enclose_area
    return gious


In [19]:
import json
import torch
from tqdm import tqdm
# import average from math
from statistics import mean

def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs

pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.75,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.5290850628471208
thresh=0.75, acc=0.39272142648348435
thresh=0.9, acc=0.15536392867582577
iou|0.5:0.9=0.39824287895027444
tensor([0.9647, 0.9647, 0.0128,  ..., 0.5212, 0.0000, 0.0000])
{0.5: 0.5290850628471208, 0.55: 0.5137386729026601, 0.6: 0.4937152879275066, 0.65: 0.46609178602747736, 0.7: 0.43320666471791874, 0.75: 0.39272142648348435, 0.8: 0.33791289096755334, 0.85: 0.26235019000292314, 0.9: 0.15536392867582577}
6842/6842


In [56]:
for annt,pred_bbox,iou_i in zip(all_annts,pred_bboxes,iou):
    annt[f'{model}_{mode}_pred']=dict(
        # save as list
        pred_bbox=pred_bbox.tolist(),
        iou=float(iou_i)
    )
all_annts


[{'annt_id': '0000000',
  'bbox': [315.54, 56.12, 323.02, 384.14],
  'bbox_area': 36959.305749999985,
  'caption': 'The individual in question appears to be a woman dressed in a black long-sleeve top with a detailed lace-like design on the back. She is wearing a white skirt that ends above the knee, paired with black leggings and distinctive black shoes with pink soles. She seems to be standing indoors, possibly in a queue, and her posture suggests that she may be waiting for her turn or engaged in conversation with someone not visible in the closely cropped image. There is no visible text on her clothing, and she is not engaging with any objects or other people in a way that suggests a specific activity. Her overall attire gives a casual yet put-together look.',
  'category_id': 1,
  'file_name': '000000002685.jpg',
  'height': 555,
  'image_id': 'coco_000000002685',
  'recognition_category': '',
  'width': 640,
  'source': 'coco2017',
  'original_subset': 'val',
  'raw_caption': 'The

In [57]:
with open(json_path,'w') as f:
    json.dump(all_annts,f,indent=4)